In [280]:
import numpy as np
from numpy import random
from scipy.stats import norm
from scipy.stats import genextreme
import pandas as pd 
from numpy import linalg as la
from scipy import optimize

import w8_estimation as est 
import w8_LinearModel as lm
import w8_probit as probit
import w8_logit as logit

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Data exploration

In [281]:
data = pd.read_csv('ppcs_cc.csv', skiprows=1, header=None)

columns = [
    'sblack', 'shisp', 'swhite', 'sother', 'smale', 'sage',
    'sempl', 'sincome', 'spop', 'daytime', 'inctype_lin',
    'omajblack', 'omajhisp', 'omajwhite', 'omajother',
    'osplit', 'sbehavior', 'year', 'anyuseofforce_coded'
]

data.columns = columns

In [282]:
print(data.head())

   sblack  shisp  swhite  sother  smale  sage  sempl  sincome  spop  daytime  \
0       1      0       0       0      1    18      0        1     1        1   
1       1      0       0       0      1    20      1        2     4        0   
2       1      0       0       0      1    22      1        2     3        1   
3       1      0       0       0      1    22      1        3     1        1   
4       1      0       0       0      1    22      1        1     1        1   

   inctype_lin  omajblack  omajhisp  omajwhite  omajother  osplit  sbehavior  \
0            2          0         0          1          0       0          0   
1            2          0         0          1          0       0          0   
2            2          0         0          1          0       0          0   
3            2          0         0          1          0       0          0   
4            2          0         0          1          0       0          0   

   year  anyuseofforce_coded  
0  2011

## Checking reference variabels

In [283]:
data[["sblack","shisp","swhite","sother"]].sum(axis=1).eq(1).all()

np.True_

In [284]:
data[[ 'omajblack', 'omajhisp', 'omajwhite', 'omajother','osplit']].sum(axis=1).eq(1).all()

np.True_

swhite and omajwhite are removed as reference categories

In [285]:
data2 = data.drop(columns=['swhite','omajwhite'])

Recoding inctype to dummy variabels

In [286]:
data2["Street_stop"] = (data2["inctype_lin"] == 1).astype(int)
data2["Traffic_stop"] = (data2["inctype_lin"] == 2).astype(int)
data2["Other_stop"] = (data2["inctype_lin"] == 3).astype(int)
data2=data2.drop(columns=['inctype_lin'])   

Checking reference categories

In [287]:
data2[["Street_stop", "Traffic_stop", "Other_stop"]].sum(axis=1).eq(1).all()

np.True_

Removing street_stop to be used as reference

In [288]:
data2=data2.drop(columns=['Street_stop'])

## Year variable 

From an initial look at the data it seems that all observations are from 2011 and therefore the year variable should not be included. This is first investegated

In [289]:
(data2["year"] == 2011).all()

np.True_

All observations are from 2011 so the year variable is deleted.

In [290]:
data2=data2.drop(columns=['year'])

## Cheking for other constant variabels

In [291]:
data2.nunique()

sblack                  2
shisp                   2
sother                  2
smale                   2
sage                   74
sempl                   2
sincome                 3
spop                    4
daytime                 2
omajblack               2
omajhisp                2
omajother               2
osplit                  1
sbehavior               2
anyuseofforce_coded     2
Traffic_stop            2
Other_stop              1
dtype: int64

In [292]:
print(data2["osplit"].unique())
print(data2["Other_stop"].unique())

[0]
[0]


Observe that both osplit and other_stop are constant (they never occur). We remove these from the dataset

In [293]:
data2=data2.drop(columns=['osplit','Other_stop'])

## Cheking missing values

In [294]:
data2.isna().sum().sum()

np.int64(0)

No missing values

## Exploring data further and checking for seperation

In [295]:
data2.describe(include='all')

,sblack,shisp,sother,smale,sage,sempl,sincome,spop,daytime,omajblack,omajhisp,omajother,sbehavior,anyuseofforce_coded,Traffic_stop
count,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000
mean,0.110555,0.101606,0.048697,0.529613,41.010003,0.695446,2.164780,1.362727,0.666491,0.060805,0.023954,0.011582,0.065017,0.005001,0.958410
std,0.313622,0.302169,0.215262,0.499188,16.146916,0.460279,0.848262,0.765598,0.471529,0.239005,0.152925,0.107009,0.246589,0.070552,0.199676
min,0.000000,0.000000,0.000000,0.000000,16.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,27.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,0.000000,0.000000,0.000000,1.000000,40.000000,1.000000,2.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,1.000000,52.000000,1.000000,3.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,90.000000,1.000000,3.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


The mean of anyuseofforce is very low, indicating that only in very few of the observations any force where used. We explore the number of observations.

In [296]:
(data2['anyuseofforce_coded']==1).sum()

np.int64(19)

Only 19 of the observations in the dataset where force was used. This is a limited amount of observations. We explore these observations further to check for seperation.

In [297]:
data_forceused=data2[data2['anyuseofforce_coded']==1]
data_forceused.describe(include='all')

,sblack,shisp,sother,smale,sage,sempl,sincome,spop,daytime,omajblack,omajhisp,omajother,sbehavior,anyuseofforce_coded,Traffic_stop
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.0,19.000000,19.0,19.000000,19.0,19.000000
mean,0.157895,0.315789,0.052632,0.789474,30.789474,0.473684,2.052632,1.947368,0.473684,0.0,0.052632,0.0,0.526316,1.0,0.684211
std,0.374634,0.477567,0.229416,0.418854,11.193147,0.512989,0.848115,1.078769,0.512989,0.0,0.229416,0.0,0.512989,0.0,0.477567
min,0.000000,0.000000,0.000000,0.000000,18.000000,0.000000,1.000000,1.000000,0.000000,0.0,0.000000,0.0,0.000000,1.0,0.000000
25%,0.000000,0.000000,0.000000,1.000000,23.500000,0.000000,1.000000,1.000000,0.000000,0.0,0.000000,0.0,0.000000,1.0,0.000000
50%,0.000000,0.000000,0.000000,1.000000,28.000000,0.000000,2.000000,2.000000,0.000000,0.0,0.000000,0.0,1.000000,1.0,1.000000
75%,0.000000,1.000000,0.000000,1.000000,33.000000,1.000000,3.000000,2.000000,1.000000,0.0,0.000000,0.0,1.000000,1.0,1.000000
max,1.000000,1.000000,1.000000,1.000000,59.000000,1.000000,3.000000,4.000000,1.000000,0.0,1.000000,0.0,1.000000,1.0,1.000000


Notice that none of the observations where force was used, had a majority of black officers or "other" race. This would create a problem of seperation if the variabels where kept as is. We therefore consider to colapse indicator variabels of race of majority of officers such that there is just one for all non white majorities. But we observe:

In [298]:
(data_forceused['omajhisp']==1).sum()

np.int64(1)

That is there is only 1 observation where force was used where the majority of officers where non white (around 0.05%). This is very limited, however in the whole dataset we obserse that only in around 1% of the observations in the dataset the majority of officers where non white, so this is not that surprising. We choose to keep the split between white and non white majority of officers, but put keeping in mind that the estimate of the parameter is relativly uncertain, but this estimator not the main focus on the assigment. 

In [299]:
#Recoding of the majority race variable.
data2["omajnonwhite"] = ((data2["omajblack"] == 1) | (data2["omajhisp"] == 1) | (data2["omajother"] == 1)).astype(int)
data2=data2.drop(columns=['omajblack','omajhisp','omajother'])


Efter at have snakket mere med nogle af de andre til øvelser idag tror jeg måske jeg synes vi skal fjerne den helt for det giver vidst nogle problemer med bootstrapping senere at beholde den.

In [300]:
data2= data2.drop(columns=['omajnonwhite'])

## Rescaling age variable

In the initial data exploration we observe that the age variable takes values on a scale from 16 to 90, with a mean of 41. The BFGS algorithm used later for optimizing can be sensitive to this so we use a rescaling instead.

In [301]:
data2['sage_rescaled'] = data2['sage']/10
data2=data2.drop(columns=['sage'])

## Defining X and Y and cheking rank of X

In [302]:
np.shape(data2)

(3799, 12)

Adding a constant and checking the rank of X

In [303]:
N=np.shape(data2)[0]
x_lab= ['const','sblack', 'shisp', 'sother', 'smale', 'sage_rescaled', 'sempl', 'sincome', 'spop', 'daytime','sbehavior','Traffic_stop']
y_lab= 'anyuseofforce_coded'
data2['const']=np.ones((N,))
y = data2[y_lab].values 
x = data2[x_lab].values
K=np.shape(x)[1]

In [304]:
np.shape(x)

(3799, 12)

In [305]:
print(la.matrix_rank(x))

12


X has full rank.

# Modelling - estimating parameters

## Linear model

In [306]:
ols_results = lm.estimate(y, x, robust_se=True)
ols_tab = lm.print_table((y_lab, x_lab), ols_results, title='LPM results')
ols_tab

LPM results
Dependent variable: anyuseofforce_coded

R2 = 0.033
sigma2 = nan


,b_hat,se,t
const,0.0306,0.0141,2.1672
sblack,0.0014,0.0044,0.3259
shisp,0.0088,0.0059,1.4994
sother,-0.0005,0.0058,-0.0939
smale,0.0042,0.0022,1.8873
sage_rescaled,-0.0016,0.0006,-2.5849
sempl,-0.0056,0.0031,-1.8052
sincome,0.0009,0.0014,0.6847
spop,0.0040,0.0021,1.8913
daytime,-0.0018,0.0029,-0.6177


## Probit model

In [318]:
theta0 = probit.starting_values(y, x)
ll = probit.loglikelihood(theta0, y, x)
probit_results = est.estimate(probit.q, theta0, y, x)

Optimization terminated successfully.
         Current function value: 0.022861
         Iterations: 69
         Function evaluations: 975
         Gradient evaluations: 75


In [319]:
probit_tab = est.print_table(x_lab, probit_results, title=f'Probit, y = {y_lab}')
probit_tab

Optimizer succeeded after 69 iter. (975 func. evals.). Final criterion:  0.02286.
Probit, y = anyuseofforce_coded


,theta,se,t
const,-2.4090,1.1580,-2.0803
sblack,0.1814,0.3580,0.5068
shisp,0.4029,0.2963,1.3598
sother,0.0609,0.6247,0.0975
smale,0.5708,0.3713,1.5375
sage_rescaled,-0.1284,0.1862,-0.6897
sempl,-0.4064,0.2750,-1.4774
sincome,0.0930,0.1873,0.4966
spop,0.1899,0.1504,1.2627
daytime,-0.1349,0.2754,-0.4899


## Logit model

In [309]:
theta0 = logit.starting_values(y, x)
logit_results = est.estimate(logit.q, theta0, y, x)

Optimization terminated successfully.
         Current function value: 0.022997
         Iterations: 117
         Function evaluations: 1547
         Gradient evaluations: 119


In [310]:
logit_tab = est.print_table(x_lab, logit_results, title=f'Logit, y = {y_lab}')
logit_tab

Optimizer succeeded after 117 iter. (1547 func. evals.). Final criterion:    0.023.
Logit, y = anyuseofforce_coded


,theta,se,t
const,-4.7237,2.3962,-1.9713
sblack,0.3333,0.9036,0.3689
shisp,0.8847,0.6967,1.2698
sother,-0.2666,1.5329,-0.1739
smale,1.1872,0.8001,1.4838
sage_rescaled,-0.3705,0.4111,-0.9012
sempl,-0.9839,0.5973,-1.6473
sincome,0.1909,0.4051,0.4713
spop,0.4914,0.3169,1.5508
daytime,-0.3757,0.6295,-0.5968


## Comparison of estimats

In [311]:
print('Probit / OLS')
print(probit_tab.theta / ols_tab.b_hat)

Probit / OLS
const            -78.725490
sblack           129.571429
shisp             45.784091
sother          -121.800000
smale            135.904762
sage_rescaled     80.250000
sempl             72.571429
sincome          103.333333
spop              47.475000
daytime           74.944444
sbehavior         29.677966
Traffic_stop      24.844523
dtype: float64


Til øveslerne så vi at de værdier var tæt på 2.5, men det gælder kun når man ser på data hvor sucess er ca. 50%. Men det er det slet ikke her.

In [312]:
print('Logit / OLS')
print(logit_tab.theta / ols_tab.b_hat)

Logit / OLS
const           -154.369281
sblack           238.071429
shisp            100.534091
sother           533.200000
smale            282.666667
sage_rescaled    231.562500
sempl            175.696429
sincome          212.111111
spop             122.850000
daytime          208.722222
sbehavior         73.200565
Traffic_stop      51.420495
dtype: float64


Samme som ovenfor

In [313]:
print('Logit / Probit')
print(logit_tab.theta / probit_tab.theta)

Logit / Probit
const            1.960855
sblack           1.837376
shisp            2.195830
sother          -4.377668
smale            2.079888
sage_rescaled    2.885514
sempl            2.421014
sincome          2.052688
spop             2.587678
daytime          2.785026
sbehavior        2.466495
Traffic_stop     2.069691
Name: theta, dtype: float64


Det samme her vi kan heller ikke forvente den er omkring 1.6